# 4h. Key findings and Step 4 questions for management

This notebook reads the Step 3 and Step 4 output tables, collects the headline numbers into `outputs/tables/key_figures.csv` (the single source the databook, memo and README draw on), writes `outputs/key_findings.md`, and adds the questions the analyses raised to the Q&A log. No number here is typed in: every figure is read from `outputs/tables/`.

In [1]:
import pandas as pd
import numpy as np
from nb_utils import connect, q, read_table, save_table, render_qa_log, gbp, ROOT

T = {name: read_table(name) for name in [
    "4a_revenue_mix", "4a_arr", "4a_growth", "4a_uplift_factors", "4b_top_n_shares", "4b_large_account_share", "4b_top10_runrate",
    "4b_declining_customer_lines", "4c_old_vs_new_cohorts", "4c_tenure_matched", "4d_nrr_grr", "4e_arr_bridge", "4e_revenue_bridge",
    "4f_margin_by_line", "4f_margin_reconciliation", "4f_mix_rate", "4f_discount_by_year", "4f_discount_by_size", "4g_segments",
    "4g_retention_12m_by_segment", "4g_company_benchmarks", "step3_reported_total_effects", "step3_recon_annual", "step2_overdue_by_year"]}

def v(table, col, **where):
    d = T[table]
    for k_, val in where.items():
        d = d[d[k_] == val]
    assert len(d) == 1, (table, col, where, len(d))
    return float(d[col].iloc[0])

## 1. Key figures

In [2]:
kf = []
def add(name, value, unit, source, note=""):
    kf.append({"name": name, "value": value, "unit": unit, "source": source, "note": note})

for y in (2022, 2023, 2024, 2025):
    add(f"net_revenue_{y}", v("4a_revenue_mix", "net_revenue", year=y), "gbp", "4a_revenue_mix")
    add(f"recurring_pct_{y}", v("4a_revenue_mix", "recurring_pct", year=y), "pct", "4a_revenue_mix")
    add(f"arr_{y}", v("4a_arr", "arr", year=y), "gbp", "4a_arr")
    add(f"mgmt_revenue_{y}", v("step3_recon_annual", "mgmt_total_revenue", year=y), "gbp", "step3_recon_annual")
    add(f"gross_margin_{y}", v("4f_margin_reconciliation", "gross_margin_pct", year=y), "pct", "4f_margin_reconciliation")
    add(f"overdue_pct_{y}", v("step2_overdue_by_year", "overdue_pct", year=y), "pct", "step2_overdue_by_year")
for y in (2023, 2024, 2025):
    add(f"headline_growth_{y}", v("4a_growth", "headline_growth_pct", year=y), "pct", "4a_growth")
    add(f"recurring_growth_{y}", v("4a_growth", "recurring_growth_pct", year=y), "pct", "4a_growth")
    add(f"growth_ex_other_implementation_{y}", v("4a_growth", "growth_ex_other_implementation_pct", year=y), "pct", "4a_growth")
    add(f"arr_growth_{y}", v("4a_arr", "arr_growth_pct", year=y), "pct", "4a_arr")
    add(f"arr_growth_ex_price_{y}", v("4a_arr", "arr_growth_ex_price_pct", year=y), "pct", "4a_arr")
    add(f"price_uplift_{y}", 100 * (v("4a_uplift_factors", "k", year=y) - 1), "pct", "4a_uplift_factors")
    add(f"nrr_{y}", v("4d_nrr_grr", "nrr_pct", year=y, basis="reported"), "pct", "4d_nrr_grr")
    add(f"grr_{y}", v("4d_nrr_grr", "grr_pct", year=y, basis="reported"), "pct", "4d_nrr_grr")
    add(f"nrr_ex_price_{y}", v("4d_nrr_grr", "nrr_pct", year=y, basis="excluding price increase"), "pct", "4d_nrr_grr")
    add(f"nrr_ex_top5_{y}", v("4d_nrr_grr", "nrr_pct", year=y, basis="excluding top 5 customers"), "pct", "4d_nrr_grr")
    add(f"churned_customers_{y}", v("4e_arr_bridge", "churned_customers", year=y), "count", "4e_arr_bridge")
    add(f"logo_churn_rate_{y}", 100 * v("4e_arr_bridge", "churned_customers", year=y)
        / (v("4e_arr_bridge", "churned_customers", year=y) + v("4e_arr_bridge", "continuing_customers", year=y)), "pct", "4e_arr_bridge")
    for comp in ("new", "cross_sell", "upsell", "price_increase", "downgrade", "contraction", "churn"):
        add(f"arr_bridge_{comp}_{y}", v("4e_arr_bridge", comp, year=y), "gbp", "4e_arr_bridge")
    for comp in ("new_customers", "full_year_effect_of_prior_year_additions", "cross_sell", "upsell", "price_increase", "billing_gaps",
                 "downgrade", "contraction", "churn", "credit_notes_change", "signup_implementation_change", "other_implementation_change"):
        add(f"rev_bridge_{comp}_{y}", v("4e_revenue_bridge", comp, year=y), "gbp", "4e_revenue_bridge")
    add(f"mix_effect_{y}", v("4f_mix_rate", "mix_effect_pts", year=y), "pts", "4f_mix_rate")
    add(f"rate_effect_{y}", v("4f_mix_rate", "rate_effect_pts", year=y), "pts", "4f_mix_rate")
    add(f"manual_adjustments_{y}", v("step3_reported_total_effects", "manual_adjustments_in_total", year=y), "gbp", "step3_reported_total_effects")
add("other_implementation_2025", v("4a_revenue_mix", "other_implementation", year=2025), "gbp", "4a_revenue_mix")
add("other_implementation_pct_2025", v("4a_revenue_mix", "other_implementation_pct", year=2025), "pct", "4a_revenue_mix")
for y in (2022, 2025):
    add(f"large_account_share_{y}", v("4b_large_account_share", "large_share_of_net_pct", year=y), "pct", "4b_large_account_share")
for n in (1, 5, 10, 20):
    add(f"top{n}_share_2025", v("4b_top_n_shares", "share_of_net_pct", year=2025, top_n=n), "pct", "4b_top_n_shares")
add("top1_share_2022", v("4b_top_n_shares", "share_of_net_pct", year=2022, top_n=1), "pct", "4b_top_n_shares")
for cid in T["4b_top10_runrate"][T["4b_top10_runrate"].declining].customer_id:
    add(f"{cid}_dec_arr_2024", v("4b_top10_runrate", "dec_arr_prior", customer_id=cid), "gbp", "4b_top10_runrate")
    add(f"{cid}_dec_arr_2025", v("4b_top10_runrate", "dec_arr_latest", customer_id=cid), "gbp", "4b_top10_runrate")
    add(f"{cid}_arr_change", v("4b_top10_runrate", "change_vs_prior_dec", customer_id=cid), "gbp", "4b_top10_runrate")
    add(f"{cid}_arr_change_pct", v("4b_top10_runrate", "change_vs_prior_dec_pct", customer_id=cid), "pct", "4b_top10_runrate")
    add(f"{cid}_change_vs_peak_pct", v("4b_top10_runrate", "change_vs_peak_pct", customer_id=cid), "pct", "4b_top10_runrate")
for T_ in (6, 12, 18):
    for grp, key in (("2017-2022 signups", "older"), ("2023+ signups", "newer")):
        add(f"retention_{T_}m_{key}", v("4c_old_vs_new_cohorts", "logo_retention_pct", group=grp, tenure_months=T_), "pct", "4c_old_vs_new_cohorts")
add("retention_12m_2024_cohort", v("4c_tenure_matched", "logo_retention_pct", signup_year=2024, tenure_months=12), "pct", "4c_tenure_matched")
add("retention_12m_2022_cohort", v("4c_tenure_matched", "logo_retention_pct", signup_year=2022, tenure_months=12), "pct", "4c_tenure_matched")
add("implementation_margin_2025", v("4f_margin_by_line", "gross_margin_pct", year=2025, product_line="Implementation Services"), "pct", "4f_margin_by_line")
add("implementation_cost_2024", v("4f_margin_by_line", "cost_of_delivery", year=2024, product_line="Implementation Services"), "gbp", "4f_margin_by_line")
add("implementation_cost_2025", v("4f_margin_by_line", "cost_of_delivery", year=2025, product_line="Implementation Services"), "gbp", "4f_margin_by_line")
add("discount_given_up_2025", v("4f_discount_by_year", "revenue_given_up", year=2025), "gbp", "4f_discount_by_year")
add("discount_pct_of_list_2025", v("4f_discount_by_year", "discount_pct_of_list", year=2025), "pct", "4f_discount_by_year")
add("large_discount_pct_2025", v("4f_discount_by_size", "discount_pct_of_list", year=2025, company_size="Large (250+)"), "pct", "4f_discount_by_size")
add("large_share_of_discount_2025", v("4f_discount_by_size", "share_of_revenue_given_up_pct", year=2025, company_size="Large (250+)"), "pct", "4f_discount_by_size")
for seg in ("Paid Search", "Partner/Reseller", "Direct Sales"):
    add(f"logo_churn_2025_{seg}", v("4g_segments", "logo_churn_2025_pct", dimension="acquisition_channel", segment=seg), "pct", "4g_segments")
add("nrr_2025_Healthcare", v("4g_segments", "nrr_2025_pct", dimension="industry", segment="Healthcare"), "pct", "4g_segments")
add("retention_12m_micro_older", v("4g_retention_12m_by_segment", "retention_12m_2017_2022_pct", dimension="company_size", segment="Micro (1-9)"), "pct", "4g_retention_12m_by_segment")
add("retention_12m_micro_newer", v("4g_retention_12m_by_segment", "retention_12m_2023_plus_pct", dimension="company_size", segment="Micro (1-9)"), "pct", "4g_retention_12m_by_segment")

anom = q(connect(), "SELECT sum(-amount) AS a, count(*) AS n FROM clean_invoices WHERE anomaly_flag").iloc[0]
dup = q(connect(), "SELECT sum(amount) AS a, count(*) AS n FROM clean_removed_invoices").iloc[0]
oth = q(connect(), "SELECT count(*) AS n, sum(amount) FILTER (WHERE status = 'overdue') AS overdue FROM clean_invoices WHERE oneoff_category = 'other_implementation'").iloc[0]
add("anomaly_probable_understatement", float(anom.a), "gbp", "clean_invoices (anomaly_flag)")
add("anomaly_invoices", float(anom.n), "count", "clean_invoices (anomaly_flag)")
add("duplicates_removed", float(dup.a), "gbp", "clean_removed_invoices")
add("duplicate_rows", float(dup.n), "count", "clean_removed_invoices")
add("other_implementation_invoices", float(oth.n), "count", "clean_invoices (oneoff_category)")
add("other_implementation_overdue", float(oth.overdue), "gbp", "clean_invoices (oneoff_category)")
for y in (2022, 2023, 2024, 2025):
    add(f"recurring_revenue_{y}", v("4a_revenue_mix", "recurring", year=y), "gbp", "4a_revenue_mix")
    add(f"oneoff_revenue_{y}", v("4a_revenue_mix", "oneoff", year=y), "gbp", "4a_revenue_mix")
add("net_ex_other_implementation_2025", v("4a_growth", "net_ex_other_implementation", year=2025), "gbp", "4a_growth")
_k = {r["name"]: r["value"] for r in kf}
add("revenue_increase_2025", _k["net_revenue_2025"] - _k["net_revenue_2024"], "gbp", "derived: net_revenue_2025 - net_revenue_2024")
add("arr_increase_2025", _k["arr_2025"] - _k["arr_2024"], "gbp", "derived: arr_2025 - arr_2024")
add("reported_less_cube_2025", _k["mgmt_revenue_2025"] - _k["net_revenue_2025"], "gbp", "derived: mgmt_revenue_2025 - net_revenue_2025")
cnt = q(connect(), "SELECT (SELECT count(*) FROM raw_invoices) AS inv, (SELECT count(*) FROM raw_customers) AS cust").iloc[0]
add("raw_invoice_rows", float(cnt.inv), "count", "raw_invoices")
extra = q(connect(), """
    SELECT (SELECT count(*) FROM prof_name_shared_pairs) AS name_pairs,
           (SELECT median(amount) FROM clean_invoices WHERE oneoff_category = 'signup_implementation') AS signup_median,
           (SELECT flag_gbp FROM cfg_materiality) AS flag_gbp,
           (SELECT 100 * material_pct FROM cfg_materiality) AS monthly_pct,
           (SELECT 100 * annual_pct FROM cfg_materiality) AS annual_pct
""").iloc[0]
add("name_pairs", float(extra.name_pairs), "count", "prof_name_shared_pairs")
add("signup_implementation_median", float(extra.signup_median), "gbp", "clean_invoices (signup_implementation)")
add("materiality_flag_gbp", float(extra.flag_gbp), "gbp", "cfg_materiality")
add("materiality_monthly_pct", float(extra.monthly_pct), "pct", "cfg_materiality")
add("materiality_annual_pct", float(extra.annual_pct), "pct", "cfg_materiality")
add("customers_total", float(cnt.cust), "count", "raw_customers")
kf = save_table(pd.DataFrame(kf), "key_figures")
K = dict(zip(kf.name, kf.value))
print(len(kf), "key figures")

191 key figures


## 2. Key findings

In [3]:
m = lambda x, dp=2: f"£{x / 1e6:,.{dp}f}m"
g = lambda x: f"£{x:,.0f}"
p = lambda x, dp=1: f"{x:.{dp}f}%"
def adj(x):  # reported total against the invoices
    return f"{g(abs(x))} {'above' if x > 0 else 'below'}"
decl = [c for c in T["4b_top10_runrate"][T["4b_top10_runrate"].declining].customer_id]
lost_lines = T["4b_declining_customer_lines"]
def lines_lost(cid):
    d = lost_lines[(lost_lines.customer_id == cid) & (lost_lines["2025"] == 0)]
    return " and ".join(f"{r.product_line} (ended {r.end_date})" for r in d.itertuples())

findings = [
    ("F01", 2, "High", "4a",
     f"One-off revenue flatters 2025 growth. Headline revenue growth is {p(K['headline_growth_2025'])}; recurring revenue grew {p(K['recurring_growth_2025'])} and "
     f"revenue excluding the {K['other_implementation_invoices']:.0f} 'other implementation' invoices billed in 2025 grew {p(K['growth_ex_other_implementation_2025'])}. "
     f"Those invoices total {m(K['other_implementation_2025'])} ({p(K['other_implementation_pct_2025'])} of 2025 revenue), all to existing customers, none in the customer's signup month. "
     f"Recurring share fell from {p(K['recurring_pct_2024'])} in 2024 to {p(K['recurring_pct_2025'])}."),
    ("F02", 6, "High", "4e",
     f"Price increases carry a large part of recurring growth. Every recurring line was billed {p(K['price_uplift_2024'], 0)} more from January 2024 and a further {p(K['price_uplift_2025'], 0)} from January 2025. "
     f"ARR grew {p(K['arr_growth_2025'])} in 2025, {p(K['arr_growth_ex_price_2025'])} before the price increase ({p(K['arr_growth_2024'])} and {p(K['arr_growth_ex_price_2024'])} in 2024). "
     f"The price increase added {m(K['arr_bridge_price_increase_2025'])} of ARR in 2025; upsell was {g(K['arr_bridge_upsell_2025'])}."),
    ("F03", 3, "High", "4d",
     f"Retention is weakening. NRR was {p(K['nrr_2023'])}, {p(K['nrr_2024'])} and {p(K['nrr_2025'])} for 2023 to 2025; excluding the price increase it was "
     f"{p(K['nrr_ex_price_2023'])}, {p(K['nrr_ex_price_2024'])} and {p(K['nrr_ex_price_2025'])}. GRR fell from {p(K['grr_2023'])} to {p(K['grr_2025'])}. "
     f"Logo churn rose from {K['churned_customers_2023']:.0f} customers ({p(K['logo_churn_rate_2023'])}) in 2023 to {K['churned_customers_2025']:.0f} ({p(K['logo_churn_rate_2025'])}) in 2025. NRR and GRR are available for 2023 to 2025 only."),
    ("F04", 3, "High", "4c",
     f"Newer customers leave faster. At the same tenure, {p(K['retention_12m_newer'])} of customers who signed up from 2023 were still subscribed after 12 months, against {p(K['retention_12m_older'])} for 2017 to 2022 signups "
     f"({p(K['retention_12m_2024_cohort'])} for the 2024 cohort). The fall appears in every channel, size band and industry; it is largest for Micro customers "
     f"({p(K['retention_12m_micro_older'])} to {p(K['retention_12m_micro_newer'])})."),
    ("F05", 4, "High", "4b",
     f"Six large accounts carry {p(K['large_account_share_2025'])} of 2025 revenue ({p(K['large_account_share_2022'])} in 2022), and two of them are shrinking. "
     + " ".join(f"{cid}'s December ARR fell {g(-K[f'{cid}_arr_change'])} (down {p(-K[f'{cid}_arr_change_pct'])}) to {m(K[f'{cid}_dec_arr_2025'])} after it dropped {lines_lost(cid)}." for cid in decl)
     + f" The largest customer was {p(K['top1_share_2025'])} of 2025 revenue; no customer exceeded 10% in any year."),
    ("F06", 5, "Medium", "4f",
     f"Gross margin fell from {p(K['gross_margin_2024'])} to {p(K['gross_margin_2025'])} in 2025, entirely through mix: the mix effect was {K['mix_effect_2025']:+.2f} points and the rate effect {K['rate_effect_2025']:+.2f}. "
     f"Implementation Services earns {p(K['implementation_margin_2025'])}, and its cost of delivery rose from {m(K['implementation_cost_2024'])} to {m(K['implementation_cost_2025'])}, so the 2025 implementation revenue came with its delivery cost."),
    ("F07", 5, "Medium", "4f",
     f"Discounts gave up {m(K['discount_given_up_2025'])} of list-price revenue in 2025 ({p(K['discount_pct_of_list_2025'])} of list). Large customers average {p(K['large_discount_pct_2025'])} off list and account for {p(K['large_share_of_discount_2025'])} of the revenue given up."),
    ("F08", 1, "Low", "Reconciliation",
     f"Revenue is real and reconciles. The cleaned invoice ledger ties to the management accounts' product lines in all 48 months. Reported total revenue carries four unexplained round-number adjustments: "
     f"the reported total is {adj(K['manual_adjustments_2023'])} the invoices in 2023, {adj(K['manual_adjustments_2024'])} in 2024 and {adj(K['manual_adjustments_2025'])} in 2025. {K['duplicate_rows']:.0f} duplicate invoices ({g(K['duplicates_removed'])}) "
     f"were removed from the data-room extract, and {K['anomaly_invoices']:.0f} negative paid invoices suggest reported revenue is understated by {g(K['anomaly_probable_understatement'])} over four years."),
    ("F09", 3, "Medium", "4g",
     f"Paid Search and Partner/Reseller customers churn fastest ({p(K['logo_churn_2025_Paid Search'])} and {p(K['logo_churn_2025_Partner/Reseller'])} in 2025, against {p(K['logo_churn_2025_Direct Sales'])} for Direct Sales). "
     f"Healthcare NRR was {p(K['nrr_2025_Healthcare'])} in 2025, driven by one large customer."),
    ("F10", 6, "Medium", "4e",
     f"2025 revenue rose {m(K['net_revenue_2025'] - K['net_revenue_2024'])}: other implementation {m(K['rev_bridge_other_implementation_change_2025'])}, new customers {m(K['rev_bridge_new_customers_2025'])}, "
     f"full-year effect of 2024 additions {m(K['rev_bridge_full_year_effect_of_prior_year_additions_2025'])}, price {m(K['rev_bridge_price_increase_2025'])} and cross-sell {m(K['rev_bridge_cross_sell_2025'])}, "
     f"less churn {m(-K['rev_bridge_churn_2025'])} and downgrades {m(-K['rev_bridge_downgrade_2025'])}."),
]
QN = {1: "Is the revenue real and does it reconcile?", 2: "How much is recurring?", 3: "Are customers staying, growing or leaving?",
      4: "Is the business dependent on a few customers?", 5: "Where does the margin come from?", 6: "What explains the change in revenue?",
      7: "What should change the price or structure?"}
kfind = pd.DataFrame(findings, columns=["finding_id", "investor_question", "materiality", "databook_tab", "finding"])
kfind["question_text"] = kfind.investor_question.map(QN)
save_table(kfind, "key_findings")
kfind[["finding_id", "investor_question", "materiality"]]

,finding_id,investor_question,materiality
0,F01,2,High
1,F02,6,High
2,F03,3,High
3,F04,3,High
4,F05,4,High
5,F06,5,Medium
6,F07,5,Medium
7,F08,1,Low
8,F09,3,Medium
9,F10,6,Medium


In [4]:
risks = [
    ("Other implementation revenue in 2025", "F01", "Price: value 2025 on recurring revenue or exclude the other implementation invoices; structure: specific warranty on the contracts behind them."),
    ("Growth dependent on price increases", "F02, F03", "Price: underwrite ARR growth before price rises; structure: earn-out tied to ARR or NRR."),
    ("Weakening retention in newer cohorts", "F03, F04", "Price: lower forecast retention; further diligence on 2023 onwards acquisition."),
    ("Contraction of the largest customers", "F05", "Price: adjust for lost run-rate; structure: warranty or escrow tied to large-account renewals."),
    ("Manual adjustments to reported revenue", "F08", "Structure: warranty on the management accounts; confirm the figure used in the information memorandum."),
]
lines = ["# Key findings", "",
         "Generated by `notebooks/04h_key_findings.ipynb` from `outputs/tables/`. Every number is read from the pipeline; do not edit by hand. "
         "Synthetic data: this is a self-directed simulation, not client work.", "",
         "Materiality: **High** could change the price or structure of the deal; **Medium** should be reflected in the investment case; **Low** is a point for the SPA or completion accounts.", ""]
for qn in range(1, 7):
    sub = kfind[kfind.investor_question == qn]
    if sub.empty:
        continue
    lines += [f"## Q{qn}. {QN[qn]}", ""]
    for r in sub.itertuples():
        lines += [f"- **{r.finding_id} ({r.materiality}, databook tab {r.databook_tab}).** {r.finding}", ""]
lines += [f"## Q7. {QN[7]}", ""]
for title, refs, action in risks:
    lines += [f"- **{title}** ({refs}). {action}", ""]
(ROOT / "outputs" / "key_findings.md").write_text("\n".join(lines), encoding="utf-8")
print("wrote outputs/key_findings.md")

wrote outputs/key_findings.md


## 3. Questions for management raised by Step 4

In [5]:
rw = connect(read_only=False)
rw.execute("DELETE FROM qa_log WHERE step = 'Step 4'")
start = int(q(rw, "SELECT max(CAST(substr(qa_id, 2) AS INTEGER)) AS n FROM qa_log").n.iloc[0]) + 1
qs = [
    ("4a", "Price increases",
     "Please confirm the price increases applied from January 2024 and January 2025, the contractual basis for them, how customers were notified, and whether a further increase is planned for 2026.",
     f"Every recurring line bills {p(K['price_uplift_2024'], 0)} above its contracted price in 2024 and a further {p(K['price_uplift_2025'], 0)} in 2025. The subscription table does not record the increases."),
    ("4b", "Large customers",
     " ".join(f"Why did {cid} end {lines_lost(cid)}? What is the renewal date and status of its remaining contract?" for cid in decl),
     " ".join(f"{cid}: December ARR {m(K[f'{cid}_dec_arr_2024'])} in 2024, {m(K[f'{cid}_dec_arr_2025'])} in 2025 (down {p(-K[f'{cid}_arr_change_pct'])})." for cid in decl)),
    ("4b", "Large customers",
     "Please provide the contracts for the six customers priced at 10 times list or more, including term, renewal dates, termination and change-of-control clauses.",
     f"They carry {p(K['large_account_share_2025'])} of 2025 revenue."),
    ("4c", "Retention",
     "What changed in customer acquisition, onboarding or pricing from 2023 that could explain faster churn among newer customers? Please provide churn reasons by customer for 2024 and 2025.",
     f"12-month logo retention: {p(K['retention_12m_newer'])} for 2023+ signups against {p(K['retention_12m_older'])} for 2017 to 2022 signups; Micro customers {p(K['retention_12m_micro_newer'])}."),
    ("4g", "Retention",
     "Why do customers acquired through Paid Search and Partner/Reseller churn faster, and what is the acquisition cost of each channel?",
     f"2025 logo churn: Paid Search {p(K['logo_churn_2025_Paid Search'])}, Partner/Reseller {p(K['logo_churn_2025_Partner/Reseller'])}, Direct Sales {p(K['logo_churn_2025_Direct Sales'])}."),
    ("4f", "Margin",
     "Is the Implementation Services cost of delivery for the 2025 projects complete, and is further cost to come on work already billed?",
     f"Implementation cost of delivery rose from {m(K['implementation_cost_2024'])} in 2024 to {m(K['implementation_cost_2025'])} in 2025; margin {p(K['implementation_margin_2025'])}."),
    ("4f", "Pricing",
     "What is the discount policy for large customers, who approves discounts, and are any discounts due to fall away or be renegotiated at renewal?",
     f"Large customers average {p(K['large_discount_pct_2025'])} off list; {m(K['discount_given_up_2025'])} given up across all customers in 2025."),
]
rows = [(f"Q{start + i:02d}", "Step 4", ref, topic, question, evidence, "open") for i, (ref, topic, question, evidence) in enumerate(qs)]
rw.executemany("INSERT INTO qa_log VALUES (?, ?, ?, ?, ?, ?, ?)", rows)
n = render_qa_log(rw)
save_table(q(rw, "SELECT * FROM qa_log ORDER BY qa_id"), "qa_log")
rw.close()
print(f"added {len(rows)} Step 4 questions; docs/qa_log.md now has {n}")

added 7 Step 4 questions; docs/qa_log.md now has 21


In [6]:
print(open(ROOT / "outputs" / "key_findings.md").read()[:3000])

# Key findings

Generated by `notebooks/04h_key_findings.ipynb` from `outputs/tables/`. Every number is read from the pipeline; do not edit by hand. Synthetic data: this is a self-directed simulation, not client work.

Materiality: **High** could change the price or structure of the deal; **Medium** should be reflected in the investment case; **Low** is a point for the SPA or completion accounts.

## Q1. Is the revenue real and does it reconcile?

- **F08 (Low, databook tab Reconciliation).** Revenue is real and reconciles. The cleaned invoice ledger ties to the management accounts' product lines in all 48 months. Reported total revenue carries four unexplained round-number adjustments: the reported total is £18,500 below the invoices in 2023, £2,500 above in 2024 and £15,000 above in 2025. 40 duplicate invoices (£96,957) were removed from the data-room extract, and 15 negative paid invoices suggest reported revenue is understated by £13,844 over four years.

## Q2. How much is recurri